# SST-2 — Independent contextual calibration replication

**Reference:** Zhao et al. (2021), *Calibrate Before Use: Improving Few-Shot Performance of Language Models*.

This notebook is an independent implementation of SST-2 few-shot classification and contextual calibration. It reads the original authors' `stsa.binary.train` and `stsa.binary.test` files from their public repository, and evaluates 300 seeded random test examples.The published authors' README documents GPT-2 XL, 0/1/4/8 shots, five seeds and a 300-example test subsample. Run cells from top to bottom


In [ ]:
%pip install pandas tqdm
%pip install matplotlib

## 1. Run settings

GPT-2 Xl in float32, 300 test examples, 5 seeds. We use APPROX=False for all runs, which retains full label probabilities. Results saved locally under independent_sst2_results/.

In [ ]:
import gc, hashlib, itertools, json, os, platform, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

SEED = 42
N_EVAL = 300
CF_INPUTS = ("N/A", "", "[MASK]")   # same set/order as the authors
AUTHOR_SEEDS = range(5)             # authors: seed in range(num_seeds), num_seeds=5
TEST_SEED = 0                       # authors: np.random.seed(0) freezes the 300-example test subsample
XL_FP32 = True                      # authors load GPT-2 in float32 (HF default). False = ~3-5x faster float16 for XL
APPROX = False                       # authors' README command uses --approx
APPROX_TOP_K = 100                  # api_num_log_prob default: label prob counted only if token is in top-100
MODEL_IDS = ("openai-community/gpt2-xl",)
PROTOCOL = "sst2-author-stsa-binary-test300-v2-authorsampling"

OUTPUT_ROOT = Path.cwd() / "independent_sst2_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Results directory:", OUTPUT_ROOT.resolve())

assert torch.cuda.is_available(), 'CUDA GPU required'
DEVICE = 'cuda'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__, 'Transformers:', transformers.__version__)

## 2. Original SST-2 files and frozen experimental design

Downloads author's `data/sst2/stsa.binary.train` and `data/sst2/stsa.binary.test` from their official public repository. Selects 300 random held-out test examples and sample demonstrations exclusively from the training file. Save configuration details


In [ ]:
import urllib.request
DATA_ROOT = OUTPUT_ROOT / "source_sst2_author"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/tonyzhaozh/few-shot-learning/main/data/sst2/"

def get_author_file(name):
    target = DATA_ROOT / name
    if not target.exists():
        try:
            urllib.request.urlretrieve(DATA_URL + name, target)
        except Exception as exc:
            raise RuntimeError(f"Cannot download original {name} from {DATA_URL}. Check internet.") from exc
    content = target.read_bytes()
    assert len(content) > 1000, f"Source file empty/invalid: {target}"
    return content

def parse_author_sst2(content, name):
    rows = []
    try:
        text = content.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = content.decode("latin-1")   # original SST files may not be valid UTF-8
    for line_no, line in enumerate(text.split("\n"), 1):
        line = line.rstrip("\r")
        if not line.strip():
            continue
        assert len(line) >= 3 and line[0] in "01" and line[1] == " ", (name, line_no, line[:80])
        sentence = line[2:].strip()
        assert sentence, (name, line_no)
        rows.append({"sentence": sentence, "label": int(line[0])})
    assert len(rows) >= N_EVAL, f"Too few rows in {name}: {len(rows)}"
    return rows

train_bytes = get_author_file("stsa.binary.train")
test_bytes = get_author_file("stsa.binary.test")
train = parse_author_sst2(train_bytes, "stsa.binary.train")
test = parse_author_sst2(test_bytes, "stsa.binary.test")
assert len(train) > len(test) >= N_EVAL
assert {r["label"] for r in train} == {0, 1}
assert {r["label"] for r in test} == {0, 1}
train_sentences = {r["sentence"] for r in train}
n_overlap = sum(r["sentence"] in train_sentences for r in test)
print(f"NOTE: {n_overlap} test sentences also appear verbatim in the training file (kept, as in the authors' code)")
assert len(test) >= N_EVAL
# Authors: np.random.seed(0); random_sampling(test, 300) -> np.random.choice(len, size, replace=False)
np.random.seed(TEST_SEED)
eval_indices = [int(v) for v in np.random.choice(len(test), size=N_EVAL, replace=False)]
eval_rows = [test[i] for i in eval_indices]
assert len(eval_rows) == N_EVAL and len(set(eval_indices)) == N_EVAL
print("Original authors' SST-2:", len(train), "train /", len(test), "test; evaluation:", N_EVAL)
print("Train SHA256:", hashlib.sha256(train_bytes).hexdigest())
print("Test SHA256:", hashlib.sha256(test_bytes).hexdigest())

N_TRAIN = len(train)
def author_demo_ids(seed, shots):
    # Authors: np.random.seed(seed); random_sampling(train, num_shots) -> np.random.choice(N, shots, replace=False)
    np.random.seed(seed)
    return [int(v) for v in np.random.choice(N_TRAIN, size=shots, replace=False)]

# Table 4: seeds 0-4 per shot count (authors run 0-shot for 5 seeds too, but it is deterministic -> one config)
configs = [{'protocol':'table4','shots':0,'set_id':0,'demo_ids':[],'order':[]}]
for shots in (1,4,8):
    for seed in AUTHOR_SEEDS:
        configs.append({'protocol':'table4','shots':shots,'set_id':seed,'demo_ids':author_demo_ids(seed,shots),'order':list(range(shots))})

assert len(configs) == 16
assert all(c['protocol'] == 'table4' for c in configs)
assert all(len(c['demo_ids']) == c['shots'] and sorted(c['order']) == list(range(c['shots'])) for c in configs)
assert all(len(set(c['demo_ids'])) == c['shots'] for c in configs)

EVAL_HASH = hashlib.sha256(json.dumps(eval_rows, sort_keys=True).encode()).hexdigest()
DESIGN = {'protocol_version':PROTOCOL, 'seed':SEED, 'n_eval':N_EVAL,
          'eval_hash':EVAL_HASH, 'eval_indices':eval_indices,
          'dataset':'tonyzhaozh/few-shot-learning:data/sst2:stsa.binary.test:random300',
          'source_sha256':{'train':hashlib.sha256(train_bytes).hexdigest(),
                           'test':hashlib.sha256(test_bytes).hexdigest()},
          'source_counts':{'train':len(train),'test':len(test)}, 'models':MODEL_IDS,
          'content_free_inputs':CF_INPUTS, 'configs':configs,
          'prompt':'Review: {sentence}\nSentiment: {label}\n\n ... Review: {test}\nSentiment:',
          'scoring':'next-token full-vocab label probs; p_cf = mean of UNNORMALIZED label probs over CF inputs, then normalized (authors); argmax(p/p_cf)',
          'approx':{'enabled':APPROX,'top_k':APPROX_TOP_K}, 'label_order':'0=Negative,1=Positive (authors)',
          'sampling':'legacy np.random: test np.random.seed(0) choice(300); demos np.random.seed(seed) choice(shots); seeds 0-4',
          'precision':{'gpt2':'float32','gpt2-xl':'float32' if XL_FP32 else 'float16'}}
RUN_ID = hashlib.sha256(json.dumps(DESIGN, sort_keys=True).encode()).hexdigest()[:16]
OUT = OUTPUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
manifest = OUT/'design.json'
if manifest.exists():
    assert json.loads(manifest.read_text()) == json.loads(json.dumps(DESIGN)), 'Design mismatch: do not mix runs'
else:
    manifest.write_text(json.dumps(DESIGN, indent=2))
print('Run ID:', RUN_ID, '| total configs/model:', len(configs))
print(pd.DataFrame(configs).groupby(['protocol','shots']).size().to_string())
print('Evaluation label counts:', pd.Series([x['label'] for x in eval_rows]).value_counts().to_dict())

## 3. Prompt construction and label scoring

Defines prompt construction, label scoring and contextual calibration

In [ ]:
LABELS = (' Negative',' Positive')   # authors' label_dict: 0=Negative, 1=Positive
def label_text(y): return 'Positive' if int(y)==1 else 'Negative'
def true_index(y): return int(y)

def prefix_for(c):
    # Same format as the authors' construct_prompt: each demonstration ends with a blank line.
    return ''.join(
        f"Review: {train[c['demo_ids'][j]]['sentence'].strip()}\nSentiment: {label_text(train[c['demo_ids'][j]]['label'])}\n\n"
        for j in c['order'])

def prompt_for(prefix, sentence):
    return prefix + f"Review: {str(sentence).strip()}\nSentiment:"

class Scorer:
    def __init__(self, model_id):
        self.model_id = model_id
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        ids = [self.tokenizer.encode(s, add_special_tokens=False) for s in LABELS]
        assert all(len(x)==1 for x in ids), f'Label not single token: {ids}'
        self.label_ids = [v[0] for v in ids]
        assert len(set(self.label_ids))==2
        self.tokenizer.padding_side = 'left'
        self.tokenizer.pad_token = self.tokenizer.eos_token
        dtype = torch.float32 if (model_id.endswith('/gpt2') or XL_FP32) else torch.float16
        self.model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(DEVICE).eval()
        self.batch_size = 32 if model_id.endswith('/gpt2') else (8 if XL_FP32 else 16)
        print('Loaded', model_id, '| dtype:', dtype, '| labels:', self.label_ids, '| batch:', self.batch_size)

    @torch.inference_mode()
    def probs_many(self, prompts, raw=False, approx=False):
        if not prompts: return np.empty((0,2),dtype=np.float64)
        pieces = []
        for start in range(0,len(prompts),self.batch_size):
            chunk = prompts[start:start+self.batch_size]
            enc = self.tokenizer(chunk, padding=True, return_tensors='pt', truncation=False)
            assert enc['input_ids'].shape[1] <= self.model.config.n_positions, 'Prompt exceeds GPT-2 context window'
            enc = {k:v.to(DEVICE) for k,v in enc.items()}
            # Correct GPT-2 position indices for left-padded batched inference.
            pos = (enc['attention_mask'].cumsum(-1)-1).clamp(min=0)
            logits = self.model(**enc, position_ids=pos).logits[:,-1,:].float()
            full = torch.softmax(logits,dim=-1)               # full-vocabulary next-token distribution
            p_lab = full[:,self.label_ids]
            if approx:
                # authors' --approx: a label's prob counts only if its token is in the top-K next tokens, else 0
                rank = (full.unsqueeze(1) > p_lab.unsqueeze(-1)).sum(-1)   # tokens strictly more probable
                p_lab = torch.where(rank < APPROX_TOP_K, p_lab, torch.zeros_like(p_lab))
            p = p_lab.cpu().numpy().astype(np.float64)
            if not raw:
                with np.errstate(invalid='ignore', divide='ignore'):
                    p = p/p.sum(axis=1,keepdims=True)     # rows with both labels missing become NaN (argmax -> class 0, as in authors' code)
            pieces.append(p)
        return np.concatenate(pieces,axis=0)

    def close(self):
        del self.model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


def content_free_prior(scorer, pfx):
    # Authors' get_p_content_free: average the UNNORMALIZED label probabilities over the CF inputs, THEN normalize.
    raw = scorer.probs_many([prompt_for(pfx, s) for s in CF_INPUTS], raw=True).mean(axis=0)
    return raw/raw.sum()

## 4. Checkpointed evaluation

Defines evaluation, result saving and resuming completed configurations

In [ ]:
def result_path(model_id): return OUT / f"{model_id.split('/')[-1]}_configs.csv"
def prediction_dir(model_id):
    p = OUT / f"{model_id.split('/')[-1]}_predictions"
    p.mkdir(exist_ok=True)
    return p

def evaluate_one(scorer, idx, c):
    pfx = prefix_for(c)
    prior = content_free_prior(scorer, pfx)
    assert np.isfinite(prior).all() and np.all(prior>0) and np.isclose(prior.sum(),1)
    probs = scorer.probs_many([prompt_for(pfx, r['sentence']) for r in eval_rows], approx=APPROX)
    assert probs.shape==(N_EVAL,2)
    n_nan = int(np.isnan(probs).any(axis=1).sum())   # both labels outside top-K (only possible with approx)
    base = probs.argmax(axis=1)
    cal = (probs/prior).argmax(axis=1)
    truth = np.asarray([true_index(r['label']) for r in eval_rows])
    pred = pd.DataFrame({'eval_index':np.arange(N_EVAL),'truth':truth,'baseline_pred':base,
                         'calibrated_pred':cal,'p_negative':probs[:,0],'p_positive':probs[:,1]})
    pred_file = prediction_dir(scorer.model_id)/f'config_{idx:03d}.csv'
    temp = pred_file.with_suffix('.tmp')
    pred.to_csv(temp,index=False)
    os.replace(temp,pred_file)
    return {'run_id':RUN_ID,'model':scorer.model_id,'config_index':idx,
            'protocol':c['protocol'],'shots':c['shots'],'set_id':c['set_id'],
            'demo_ids':json.dumps(c['demo_ids']),'order':json.dumps(c['order']),
            'n_eval':N_EVAL,'baseline_accuracy':float(np.mean(base==truth)),
            'calibrated_accuracy':float(np.mean(cal==truth)),
            'changed_predictions':int(np.sum(base!=cal)),
            'p_cf_negative':float(prior[0]),'p_cf_positive':float(prior[1]),'n_both_labels_missing':n_nan,
            'predictions_file':str(pred_file.relative_to(OUT))}

def run_model(model_id):
    csv_path = result_path(model_id)
    done = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    if len(done):
        assert done['run_id'].eq(RUN_ID).all() and done['model'].eq(model_id).all()
        assert not done['config_index'].duplicated().any()
        assert set(done['config_index']).issubset(set(range(len(configs))))
    complete = set(done['config_index'].astype(int)) if len(done) else set()
    # A completed configuration must have its predictions available too.
    for i in complete:
        assert (prediction_dir(model_id)/f'config_{i:03d}.csv').exists(), f'Missing prediction file for config {i}'
    pending = [(i,c) for i,c in enumerate(configs) if i not in complete]
    print(model_id, 'completed:',len(complete),'pending:',len(pending))
    if not pending: return done.sort_values('config_index')
    scorer = Scorer(model_id)
    try:
        for i,c in tqdm(pending,desc=model_id):
            row = evaluate_one(scorer,i,c)
            pd.DataFrame([row]).to_csv(csv_path, mode='a',header=not csv_path.exists(),index=False)
    finally:
        scorer.close()
    result = pd.read_csv(csv_path).sort_values('config_index')
    assert len(result)==len(configs)
    return result

## 5. Run GPT-2 XL

Runs the 16 configurations and saves results after each one, skips configurations already completed


In [ ]:
gc.collect()
torch.cuda.empty_cache()

results = run_model(MODEL_IDS[0])
print("GPT-2 XL done:", len(results), "configurations")

## 6. Verify and summarize results

Checks that saved predictions reproduce the reported accuracies. Reports mean accuracy, population standard deviation (ddof=0), and calibration gain in percentage points for each shot count, saves the summary as shot_summary.csv

In [ ]:
# Check that saved predictions reproduce each configuration's accuracy.
for row in results.itertuples(index=False):
    predictions = pd.read_csv(OUT / row.predictions_file)
    assert len(predictions) == N_EVAL
    assert np.isclose(
        (predictions.truth == predictions.baseline_pred).mean(),
        row.baseline_accuracy,
    )
    assert np.isclose(
        (predictions.truth == predictions.calibrated_pred).mean(),
        row.calibrated_accuracy,
    )

summary = results.groupby("shots").agg(
    configurations=("baseline_accuracy", "size"),
    original_mean=("baseline_accuracy", "mean"),
    calibrated_mean=("calibrated_accuracy", "mean"),
    original_sd=("baseline_accuracy", lambda x: x.std(ddof=0)),
    calibrated_sd=("calibrated_accuracy", lambda x: x.std(ddof=0)),
)

summary["gain_pp"] = (summary["calibrated_mean"] - summary["original_mean"]) * 100

summary.to_csv(OUT / "shot_summary.csv")
display(summary)
print("Saved predictions agree with reported accuracies.")

## 7. Plot and export

Plots the mean accuracy and population standard deviation before and after calibration and saves the figure and exports the results as a ZIP file

In [ ]:
import matplotlib.pyplot as plt
import shutil

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for column, label in [
    ("original", "Original"),
    ("calibrated", "Calibrated"),
]:
    axes[0].plot(
        summary.index, summary[f"{column}_mean"] * 100,
        marker="o", label=label,
    )
    axes[1].plot(
        summary.index, summary[f"{column}_sd"] * 100,
        marker="o", label=label,
    )

axes[0].set(
    xlabel="Demonstrations",
    ylabel="Mean accuracy (%)",
    title="GPT-2 XL on SST-2",
)
axes[1].set(
    xlabel="Demonstrations",
    ylabel="Population SD (percentage points)",
    title="Variability across demonstration seeds",
)

for ax in axes:
    ax.set_xticks([0, 1, 4, 8])
    ax.legend()

fig.tight_layout()
fig.savefig(OUT / "summary_plots.png", dpi=200)
plt.show()

archive = shutil.make_archive(
    str(OUTPUT_ROOT / f"sst2_independent_{RUN_ID}"),
    "zip",
    root_dir=OUT,
)
print("Results ZIP:", archive)